# Errors

Every kind of failure a notebook shows.  Cells after a failing cell still run (`allow_errors`), so
later cells are normal.  The traceback strings contain ANSI colour codes exactly as ipykernel sends them.

In [1]:
import sys, os, time, warnings
print("setup ok")

setup ok


## SyntaxError family

In [2]:
def broken(:
    pass

SyntaxError: invalid syntax (1850221797.py, line 1)

In [3]:
print("unclosed paren"
print("next line")

SyntaxError: '(' was never closed (2412611511.py, line 1)

In [4]:
if True:
print('no indent')

IndentationError: expected an indented block after 'if' statement on line 1 (1032367827.py, line 2)

In [5]:
1abc = 3

SyntaxError: invalid decimal literal (84099057.py, line 1)

## NameError / AttributeError / TypeError / KeyError / IndexError / FileNotFoundError

In [6]:
print(undefined_variable_name)

NameError: name 'undefined_variable_name' is not defined

In [7]:
import math
math.sinn(1.0)

AttributeError: module 'math' has no attribute 'sinn'

In [8]:
"1" + 1

TypeError: can only concatenate str (not "int") to str

In [9]:
{"a": 1}["b"]

KeyError: 'b'

In [10]:
[1, 2, 3][10]

IndexError: list index out of range

In [11]:
open("/definitely/not/here.txt")

FileNotFoundError: [Errno 2] No such file or directory: '/definitely/not/here.txt'

In [12]:
import not_a_real_module_xyz

ModuleNotFoundError: No module named 'not_a_real_module_xyz'

## A ZeroDivisionError fourteen frames deep

In [13]:
def level_01(x):
    return level_02(x + 1)
def level_02(x):
    return level_03(x + 1)
def level_03(x):
    return level_04(x + 1)
def level_04(x):
    return level_05(x + 1)
def level_05(x):
    return level_06(x + 1)
def level_06(x):
    return level_07(x + 1)
def level_07(x):
    return level_08(x + 1)
def level_08(x):
    return level_09(x + 1)
def level_09(x):
    return level_10(x + 1)
def level_10(x):
    return level_11(x + 1)
def level_11(x):
    return level_12(x + 1)
def level_12(x):
    return level_13(x + 1)
def level_13(x):
    return level_14(x + 1)
def level_14(x):
    return level_15(x + 1)
def level_15(x):
    return 100 / (x - 14)

level_01(0)

ZeroDivisionError: division by zero

## Recursion (IPython elides repeated frames)

In [14]:
def descend(n):
    if n == 0:
        return 1 / 0
    return descend(n - 1) + 1

descend(25)

ZeroDivisionError: division by zero

In [15]:
def forever(n):
    return forever(n + 1)

forever(0)

RecursionError: maximum recursion depth exceeded

## Frames inside lambda / genexpr / comprehension

In [16]:
data = [1, 2, 0, 4]
list(map(lambda v: 10 // v, data))

ZeroDivisionError: integer division or modulo by zero

In [17]:
sum(10 // v for v in data)

ZeroDivisionError: integer division or modulo by zero

## KeyboardInterrupt

In [18]:
import os, signal, time
os.kill(os.getpid(), signal.SIGINT)    # a real SIGINT delivered to the kernel mid-cell
time.sleep(3)
print("unreachable")

KeyboardInterrupt: 

In [19]:
raise KeyboardInterrupt("simulated Ctrl-C")

KeyboardInterrupt: simulated Ctrl-C

## Huge messages

In [20]:
raise ValueError("long message: " + "lorem ipsum dolor sit amet " * 800)

ValueError: long message: lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lorem ipsum dolor sit amet lo

In [21]:
raise ValueError("Validation failed:\n" + "\n".join(f"  - field_{i}: expected int, got 'str' (row {i * 7})" for i in range(150)))

ValueError: Validation failed:
  - field_0: expected int, got 'str' (row 0)
  - field_1: expected int, got 'str' (row 7)
  - field_2: expected int, got 'str' (row 14)
  - field_3: expected int, got 'str' (row 21)
  - field_4: expected int, got 'str' (row 28)
  - field_5: expected int, got 'str' (row 35)
  - field_6: expected int, got 'str' (row 42)
  - field_7: expected int, got 'str' (row 49)
  - field_8: expected int, got 'str' (row 56)
  - field_9: expected int, got 'str' (row 63)
  - field_10: expected int, got 'str' (row 70)
  - field_11: expected int, got 'str' (row 77)
  - field_12: expected int, got 'str' (row 84)
  - field_13: expected int, got 'str' (row 91)
  - field_14: expected int, got 'str' (row 98)
  - field_15: expected int, got 'str' (row 105)
  - field_16: expected int, got 'str' (row 112)
  - field_17: expected int, got 'str' (row 119)
  - field_18: expected int, got 'str' (row 126)
  - field_19: expected int, got 'str' (row 133)
  - field_20: expected int, got 'str

## stdout + stderr, then failure

In [22]:
import sys
print("starting step 1")
print("WARNING: disk almost full", file=sys.stderr)
print("step 2 ...", file=sys.stderr)
raise RuntimeError("step 2 failed after writing to stderr")

starting step 1


step 2 ...


RuntimeError: step 2 failed after writing to stderr

In [23]:
%%bash
echo "starting build"
echo "warning: something looks odd" >&2
ls /definitely/not/here
exit 3

starting build


ls: /definitely/not/here: No such file or directory


CalledProcessError: Command 'b'echo "starting build"\necho "warning: something looks odd" >&2\nls /definitely/not/here\nexit 3\n'' returned non-zero exit status 3.

## Chained, grouped and custom exceptions

In [24]:
def parse(s):
    try:
        return int(s)
    except ValueError as e:
        raise RuntimeError(f"could not parse {s!r}") from e

parse("12x")

RuntimeError: could not parse '12x'

In [25]:
try:
    1 / 0
except ZeroDivisionError:
    undefined_name      # NameError raised while handling the ZeroDivisionError

NameError: name 'undefined_name' is not defined

In [26]:
raise ExceptionGroup("multiple failures", [ValueError("bad value"), TypeError("bad type"), ExceptionGroup("nested", [KeyError("k")])])

  + Exception Group Traceback (most recent call last):
  |   File "/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/tmp/ipykernel_86606/1557005764.py", line 1, in <module>
  |     raise ExceptionGroup("multiple failures", [ValueError("bad value"), TypeError("bad type"), ExceptionGroup("nested", [KeyError("k")])])
  | ExceptionGroup: multiple failures (3 sub-exceptions)
  +-+---------------- 1 ----------------
    | ValueError: bad value
    +---------------- 2 ----------------
    | TypeError: bad type
    +---------------- 3 ----------------
    | ExceptionGroup: nested (1 sub-exception)
    +-+---------------- 1 ----------------
      | KeyError: 'k'
      +------------------------------------


In [27]:
class DomainError(Exception):
    def __init__(self, code, msg):
        super().__init__(f"[{code}] {msg}")
        self.code = code

raise DomainError(42, "Ünïcödé message 日本語 🚀 \twith a tab")

DomainError: [42] Ünïcödé message 日本語 🚀 	with a tab

In [28]:
raise Exception()

Exception: 

In [29]:
raise ValueError

ValueError: 

In [30]:
import sys
sys.exit(3)

SystemExit: 3

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [31]:
assert 1 + 1 == 3, "arithmetic is broken:\n  expected 3\n  got 2"

AssertionError: arithmetic is broken:
  expected 3
  got 2

In [32]:
b"\xff\xfe".decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [33]:
with warnings.catch_warnings():
    warnings.simplefilter("error")
    warnings.warn("promoted to an error", RuntimeWarning)

RuntimeWarning: promoted to an error

## Succeeding cells between failures

In [34]:
print("this cell runs fine after all those failures")
21 * 2

this cell runs fine after all those failures


42

## An object whose repr() raises

In [35]:
class Bad:
    def __repr__(self):
        raise RuntimeError("repr failed")

Bad()

RuntimeError: repr failed

## Hand-written error outputs (appended after execution)

These could not be produced by ipykernel; they model other kernels and tools.

In [36]:
# error output with an EMPTY traceback list

CustomError: no traceback was recorded

In [37]:
# traceback WITHOUT ANSI colour codes (ipython --colors=NoColor / old nbconvert)

ZeroDivisionError: division by zero

In [38]:
# the whole traceback packed into ONE string

ZeroDivisionError: division by zero

In [39]:
# empty ename and evalue

: 

In [40]:
# R kernel (IRkernel) error

simpleError: object 'x' not found

In [41]:
# Python 2 era traceback

IOError: [Errno 2] No such file or directory: 'x.txt'

In [42]:
# huge multi-megabyte-ish traceback frame list (2000 frames)

RecursionError: maximum recursion depth exceeded